# Download Tokamark Sample Data from S3

This notebook downloads Zarr files from a remote S3-compatible storage using shots from the tokamark data splits.

## Overview
- Uses tokamark library to get train/test/val shot splits
- Connects to S3 storage (STFC Echo)
- Downloads selected shots locally with error handling

## Requirements
```bash
pip install fsspec s3fs tokamark
```

## Configuration

In [15]:
import os
from pathlib import Path
from typing import List

# Configuration
S3_ENDPOINT = "https://s3.echo.stfc.ac.uk"
S3_BUCKET_PATH = "mast/tokamark/v1"
LOCAL_DATA_DIR = "./data"

# Create local directory if it doesn't exist
Path(LOCAL_DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"✓ Data directory: {Path(LOCAL_DATA_DIR).resolve()}")

✓ Data directory: /rds/project/rds-mOlK9qn0PlQ/ir-rous1/output/cnn-baseline/data


## Connect to S3 Storage

In [16]:
import fsspec

try:
    fs = fsspec.filesystem(
        "s3",
        client_kwargs={"endpoint_url": S3_ENDPOINT},
        anon=True  # Public data access
    )
    print("✓ Connected to S3 storage successfully")
except Exception as e:
    print(f"✗ Failed to connect to S3: {e}")
    raise

✓ Connected to S3 storage successfully


## Get Shots from Tokamark Data Splits

In [22]:
from tokamark.tools.path import RANDOM_SPLIT_TOKAMARK_DATA_SPLITS_FILE
from tokamark.data_split import get_train_test_val_shots

try:
    train_shots_, test_shots_, val_shots_ = get_train_test_val_shots(
        max_index=8,
        # shuffle=True,
        data_splits_file_path=RANDOM_SPLIT_TOKAMARK_DATA_SPLITS_FILE        
    )
    
    print("✓ Retrieved shots from tokamark data splits\n")
    print(f"  Train shots: {len(train_shots_)} - {train_shots_}")
    print(f"  Test shots:  {len(test_shots_)} - {test_shots_}")
    print(f"  Val shots:   {len(val_shots_)} - {val_shots_}")
    
except Exception as e:
    print(f"✗ Failed to get shots from tokamark: {e}")
    raise

✓ Retrieved shots from tokamark data splits

  Train shots: 8 - [21719, 29562, 27570, 25984, 26995, 20028, 30207, 28222]
  Test shots:  8 - [20509, 12860, 12475, 19849, 16008, 23715, 14227, 28248]
  Val shots:   8 - [16431, 24883, 12674, 16211, 22803, 27421, 12690, 26007]


## Select Which Splits to Download

In [23]:
# Choose which splits to download
# Set to True to include that split
include_train = True
include_test = True
include_val = True

# Combine selected shots
selected_shots = []
split_names = []

if include_train:
    selected_shots.extend(train_shots_)
    split_names.append(f"train ({len(train_shots_)})")

if include_test:
    selected_shots.extend(test_shots_)
    split_names.append(f"test ({len(test_shots_)})")

if include_val:
    selected_shots.extend(val_shots_)
    split_names.append(f"val ({len(val_shots_)})")

selected_shots = list(set(selected_shots))  # Remove duplicates
selected_shots.sort()

print(f"✓ Selected splits: {', '.join(split_names)}")
print(f"✓ Total unique shots: {len(selected_shots)}\n")
print("Shots to download:")
for i, shot in enumerate(selected_shots, 1):
    print(f"  {i:2d}. Shot {shot}")

✓ Selected splits: train (8), test (8), val (8)
✓ Total unique shots: 24

Shots to download:
   1. Shot 12475
   2. Shot 12674
   3. Shot 12690
   4. Shot 12860
   5. Shot 14227
   6. Shot 16008
   7. Shot 16211
   8. Shot 16431
   9. Shot 19849
  10. Shot 20028
  11. Shot 20509
  12. Shot 21719
  13. Shot 22803
  14. Shot 23715
  15. Shot 24883
  16. Shot 25984
  17. Shot 26007
  18. Shot 26995
  19. Shot 27421
  20. Shot 27570
  21. Shot 28222
  22. Shot 28248
  23. Shot 29562
  24. Shot 30207


## Find Shots in S3 Storage

In [26]:
try:
    files = fs.ls(S3_BUCKET_PATH)
    zarr_files = sorted([f for f in files if f.endswith(".zarr")])
    
    print(f"✓ Found {len(zarr_files)} Zarr files in S3\n")
    
    # Map shot numbers to file paths
    def get_shot_number(zarr_path: str) -> int:
        return int(zarr_path.split("/")[-1].replace(".zarr", ""))
    
    shot_to_path = {get_shot_number(f): f for f in zarr_files}
    
    # Find which selected shots exist in S3
    found_shots = []
    missing_shots = []
    
    for shot in selected_shots:
        if shot in shot_to_path:
            found_shots.append(shot)
        else:
            missing_shots.append(shot)
    
    print(f"✓ Found {len(found_shots)} shots in S3")
    if missing_shots:
        print(f"✗ Missing {len(missing_shots)} shots in S3: {missing_shots}")
    
    # Create list of files to download
    selected_files = [shot_to_path[shot] for shot in found_shots]
        
except Exception as e:
    print(f"✗ Failed to find shots in S3: {e}")
    raise

✓ Found 11573 Zarr files in S3

✓ Found 24 shots in S3


## Download Files with Progress Tracking

In [25]:
def download_file(fs, remote_path: str, local_dir: str) -> tuple[bool, str]:
    """
    Download a single file from remote storage.
    
    Args:
        fs: fsspec filesystem object
        remote_path: Full remote path to the file
        local_dir: Local directory to save to
    
    Returns:
        Tuple of (success: bool, message: str)
    """
    try:
        shot_name = remote_path.split("/")[-1]
        local_path = os.path.join(local_dir, shot_name)
        
        # Skip if already exists
        if os.path.exists(local_path):
            return True, f"Already exists: {shot_name}"
        
        fs.get(remote_path, local_path, recursive=True)
        return True, f"Downloaded: {shot_name}"
        
    except Exception as e:
        return False, f"Error: {str(e)}"


# Download files with summary
results = {"success": 0, "skipped": 0, "failed": 0, "errors": []}

print(f"Downloading {len(selected_files)} files...\n")

for i, remote_path in enumerate(selected_files, 1):
    shot_name = remote_path.split("/")[-1]
    success, message = download_file(fs, remote_path, LOCAL_DATA_DIR)
    
    status = "✓" if success else "✗"
    print(f"[{i}/{len(selected_files)}] {status} {message}")
    
    if success:
        if "Already" in message:
            results["skipped"] += 1
        else:
            results["success"] += 1
    else:
        results["failed"] += 1
        results["errors"].append(message)


[1/24] ✓ Downloaded: 12475.zarr
[2/24] ✓ Downloaded: 12674.zarr
[3/24] ✓ Downloaded: 12690.zarr
[4/24] ✓ Downloaded: 12860.zarr
[5/24] ✓ Downloaded: 14227.zarr
[6/24] ✓ Downloaded: 16008.zarr
[7/24] ✓ Downloaded: 16211.zarr
[8/24] ✓ Downloaded: 16431.zarr
[9/24] ✓ Downloaded: 19849.zarr
[10/24] ✓ Downloaded: 20028.zarr
[11/24] ✓ Downloaded: 20509.zarr
[12/24] ✓ Downloaded: 21719.zarr
[13/24] ✓ Downloaded: 22803.zarr
[14/24] ✓ Downloaded: 23715.zarr
[15/24] ✓ Downloaded: 24883.zarr
[16/24] ✓ Downloaded: 25984.zarr
[17/24] ✓ Downloaded: 26007.zarr
[18/24] ✓ Downloaded: 26995.zarr
[19/24] ✓ Downloaded: 27421.zarr
[20/24] ✓ Downloaded: 27570.zarr
[21/24] ✓ Downloaded: 28222.zarr
[22/24] ✓ Downloaded: 28248.zarr
[23/24] ✓ Downloaded: 29562.zarr
[24/24] ✓ Downloaded: 30207.zarr


## Download Summary

In [27]:
print("\n" + "="*50)
print("DOWNLOAD SUMMARY")
print("="*50)
print(f"✓ Successful:  {results['success']}")
print(f"⊘ Skipped:    {results['skipped']}")
print(f"✗ Failed:     {results['failed']}")
print("="*50)

if results['errors']:
    print("\nErrors encountered:")
    for error in results['errors']:
        print(f"  • {error}")

# List downloaded files
downloaded_files = list(Path(LOCAL_DATA_DIR).glob("*.zarr"))
print(f"\nLocal files in {LOCAL_DATA_DIR}: {len(downloaded_files)}")
if downloaded_files:
    print("\nFiles:")
    for f in sorted(downloaded_files):
        size_mb = f.stat().st_size / (1024**2)
        print(f"  • {f.name:20s} ({size_mb:3f} MB)")


DOWNLOAD SUMMARY
✓ Successful:  24
⊘ Skipped:    0
✗ Failed:     0

Local files in ./data: 50

Files:
  • 11849.zarr           (0.003906 MB)
  • 12413.zarr           (0.003906 MB)
  • 12475.zarr           (0.003906 MB)
  • 12674.zarr           (0.003906 MB)
  • 12690.zarr           (0.003906 MB)
  • 12860.zarr           (0.003906 MB)
  • 13686.zarr           (0.003906 MB)
  • 14144.zarr           (0.003906 MB)
  • 14227.zarr           (0.003906 MB)
  • 15245.zarr           (0.003906 MB)
  • 15490.zarr           (0.003906 MB)
  • 15656.zarr           (0.003906 MB)
  • 15971.zarr           (0.003906 MB)
  • 16008.zarr           (0.003906 MB)
  • 16211.zarr           (0.003906 MB)
  • 16422.zarr           (0.003906 MB)
  • 16431.zarr           (0.003906 MB)
  • 17080.zarr           (0.003906 MB)
  • 17393.zarr           (0.003906 MB)
  • 19387.zarr           (0.003906 MB)
  • 19849.zarr           (0.003906 MB)
  • 20028.zarr           (0.003906 MB)
  • 20127.zarr           (0.003906 MB)
